<a href="https://colab.research.google.com/github/gianlufiorini/deep-learning/blob/main/deep_learning_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [21]:
!pip install lightning
!pip install sacremoses

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 867.8/867.8 kB 7.4 MB/s  0:00:00


In [31]:
#Setup Step
from google.colab import drive
from lightning.pytorch.callbacks import ModelCheckpoint, TQDMProgressBar
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from sacremoses import MosesTokenizer
from collections import Counter
import os
import lightning as L
import torch
import pandas as pd
import gzip

drive.mount("/content/drive")
ckpt_dir = "/content/drive/MyDrive/project/checkpoints"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [34]:
#### Defining useful variables ####
PATH_DATA = "/content/data"
PATH_CKPT = "/content/drive/MyDrive/deep-learning-project/checkpoints"
PAD, UNK, SOS, EOS = "<pad>", "<unk>", "<sos>", "<eos>"
BATCH_SIZE = 80
EMBED_DIM = 256
NUM_HIDDEN = 512

#### Implementing useful functions and classes ####
#Reding text data into memory
def read_lines(path):
  with gzip.open(path, "rt", encoding="utf-8") as file:
    return [line.strip() for line in file]


#Tokenizers
mt_en = MosesTokenizer(lang="en")
mt_fr = MosesTokenizer(lang="fr")

def tokenize(text, mt):
  return mt.tokenize(text, return_str=False, escape=False)

#Vocabulary class
class Vocab:
  def __init__(self, token_lists, min_freq=2):
    counts = Counter(t for tkns in token_lists for t in tkns)

    self.itos = [PAD, UNK, SOS, EOS] + sorted(
        [w for w, c in counts.items() if c >= min_freq],
        key=lambda w: (-counts[w], w)
    )

    self.stoi = {w: i for i, w in enumerate(self.itos)}

    self.pad, self.unk, self.sos, self.eos = 0, 1, 2, 3

  def __len__(self):
    return len(self.itos)

  def encode(self, toks):
    return [self.stoi.get(t, self.unk) for t in toks]

  def decode(self, ids):
    out = []
    for i in ids:
        if i == self.eos: break
        if i not in (self.pad, self.sos): out.append(self.itos[i])
    return out

#TranslationDataset class, inheriting from Dataset class
class TranslationDataset(Dataset):
  def __init__(self, src_tkns, tgt_tkns, vs, vt):

      self.data = []
      for s, t in zip(src_tkns, tgt_tkns):
          src = vs.encode(s) + [vs.eos] #Adding eos token to source
          tgt = [vt.sos] + vt.encode(t) + [vt.eos] #Adding sos token and eos token to target
          self.data.append((torch.tensor(src), torch.tensor(tgt))) #Each entry is a tuple containing source sentence and target sentence

  def __len__(self):
    return len(self.data)

  def __getitem__(self, i):
    return self.data[i]


def collate(batch):
  src, tgt = zip(*batch) #creates a tuple containing the sources and a tuple containing the targets
  src_len = torch.tensor([len(s) for s in src]) #returns the number of tokens in each source
  src = pad_sequence(src, batch_first=True, padding_value=0)   # Creates a (B, Tx) tensor, where Tx is the length of longest source (including eos)
  tgt = pad_sequence(tgt, batch_first=True, padding_value=0)   # Creates a (B, Ty+1) tensor, where Ty is the length of longest target (including eos) + due to sos token
  return src, src_len, tgt[:, :-1], tgt[:, 1:]  # src, lengths, dec_input, labels


# Importing Data, Tokenization, and Vocabulary
We start by importing the "Multi30k" dataset, from the [official github repository](https://multi30k/dataset). We import the training, validation, and test datasets for English and French.

We proceed by tokenizing the text contained in the datasets through the open-source "Moses" tokenizer, which is the same one that was used in the paper. As in the paper, no lowercasing or stemming are applied.

In [10]:
#Training dataset
!wget -N -P /content/data https://github.com/multi30k/dataset/raw/refs/heads/master/data/task1/raw/train.fr.gz
!wget -N -P /content/data https://github.com/multi30k/dataset/raw/refs/heads/master/data/task1/raw/train.en.gz

#Validation dataset
!wget -N -P /content/data https://github.com/multi30k/dataset/raw/refs/heads/master/data/task1/raw/val.fr.gz
!wget -N -P /content/data https://github.com/multi30k/dataset/raw/refs/heads/master/data/task1/raw/val.en.gz

#Test dataset
!wget -N -P /content/data https://github.com/multi30k/dataset/raw/refs/heads/master/data/task1/raw/test_2016_flickr.fr.gz
!wget -N -P /content/data https://github.com/multi30k/dataset/raw/refs/heads/master/data/task1/raw/test_2016_flickr.en.gz

--2026-11-28 08:44:47--  https://github.com/multi30k/dataset/raw/refs/heads/master/data/task1/raw/train.fr.gz
Resolving github.com (github.com)... 20.27.177.113
Connecting to github.com (github.com)|20.27.177.113|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/multi30k/dataset/refs/heads/master/data/task1/raw/train.fr.gz [following]
--2026-11-28 08:44:48--  https://raw.githubusercontent.com/multi30k/dataset/refs/heads/master/data/task1/raw/train.fr.gz
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.111.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 604242 (590K) [application/octet-stream]
Saving to: ‘/content/data/train.fr.gz’

train.fr.gz         100%[===================>] 590.08K  --.-KB/s    in 0.1s    

Last-modified header missing -- time-

In [16]:
#Loading data in memory
train_en = read_lines(path=f"{PATH_DATA}/train.en.gz")
train_fr = read_lines(path=f"{PATH_DATA}/train.fr.gz")

val_en = read_lines(path=f"{PATH_DATA}/val.en.gz")
val_fr = read_lines(path=f"{PATH_DATA}/val.fr.gz")

test_en = read_lines(path=f"{PATH_DATA}/test_2016_flickr.en.gz")
test_fr = read_lines(path=f"{PATH_DATA}/test_2016_flickr.fr.gz")

In [24]:
#Tokenizing datasets
train_en_tkn = [tokenize(line, mt_en) for line in train_en]
train_fr_tkn = [tokenize(line, mt_en) for line in train_fr]

val_en_tkn = [tokenize(line, mt_en) for line in val_en]
val_fr_tkn = [tokenize(line, mt_en) for line in val_fr]

test_en_tkn = [tokenize(line, mt_en) for line in test_en]
test_fr_tkn = [tokenize(line, mt_en) for line in test_fr]

Next, we build a vocabulary for French and English from the training set in each language. The vocabulary includes all tokens seen at least twice in training data, plus "eos", "sos", "pad", and "unk" tokens; the "unk" token will be used to map any token that was not seen at least twice in training.

In [32]:
#Building vocabulary using the class created earlier
vocab_en = Vocab(train_en_tkn, min_freq=2)
vocab_fr = Vocab(train_fr_tkn, min_freq=2)

Having built the vocabularies, we instantiate the Dataset and DataLoader classes we will need to train and evaluate our models. For the DataLoader, we use a batch size of 80, like in the paper.

In [37]:
#Instantiating TranslationDataset classes
dataset_train = TranslationDataset(
    src_tkns=train_en_tkn,
    tgt_tkns=train_fr_tkn,
    vs = vocab_en,
    vt = vocab_fr
    )

dataset_val = TranslationDataset(
    src_tkns=val_en_tkn,
    tgt_tkns=val_fr_tkn,
    vs = vocab_en,
    vt = vocab_fr
    )

dataset_test = TranslationDataset(
    src_tkns=test_en_tkn,
    tgt_tkns=test_fr_tkn,
    vs = vocab_en,
    vt = vocab_fr
    )

#Instantiating DataLoader classes
load_train = DataLoader(dataset_train, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate)
load_val = DataLoader(dataset_val, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate)
load_test = DataLoader(dataset_test, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate)